# EmbeddingGemma 2 in FiftyOne

[EmbeddingGemma 2](https://huggingface.co/google/embeddinggemma-2) embeds text, images, video and audio into one 768-dimensional space. This notebook runs it as a FiftyOne remote zoo model on three datasets:

- **quickstart** (images): embeddings, text search, zero-shot classification
- **[VGGSound](https://huggingface.co/datasets/harpreetsahota/VGGSound)** (video with sound): video, audio and video+audio embeddings, text search, zero-shot classification
- **[VidMuse-V2M](https://huggingface.co/datasets/harpreetsahota/VidMuse-V2M)** (grouped music videos and their soundtracks): finding a video's music by similarity
- **[VidMuse-V2M-MCAP](https://huggingface.co/datasets/harpreetsahota/VidMuse-V2M-MCAP)** (the same clips as MCAP recordings): embeddings straight from `.mcap` files

In [ ]:
%pip install -q fiftyone "transformers>=5.18" torch av huggingface-hub umap-learn mcap mcap-protobuf-support

## Setup

Register the GitHub repo as a remote zoo model source. The weights (1.5 GB) download the first time the model loads.

In [ ]:
import fiftyone as fo
import fiftyone.brain as fob
import fiftyone.utils.huggingface as fouh
import fiftyone.zoo as foz

MODEL = "google/embeddinggemma-2"

foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/EmbeddingGemma2", overwrite=True
)

## Images

`quickstart` is 200 COCO images with ground truth boxes.

In [ ]:
dataset = foz.load_zoo_dataset("quickstart", dataset_name="quickstart-eg2")

### Compute embeddings

`media_type` picks the model class and the encoder to load (vision for images and video, audio for audio). Pass the dataset's own media type.

In [ ]:
model = foz.load_zoo_model(MODEL, media_type=dataset.media_type)

dataset.compute_embeddings(model, embeddings_field="eg2", batch_size=16)

### Search with text

Pass the zoo model name and `model_kwargs` to `compute_similarity()`. FiftyOne stores both with the index, so text typed into the App's search bar is embedded with the same model.

In [ ]:
fob.compute_similarity(
    dataset,
    embeddings="eg2",
    model=MODEL,
    model_kwargs={"media_type": "image"},
    brain_key="eg2_sim",
)

view = dataset.sort_by_similarity("a photo of an airplane", k=10, brain_key="eg2_sim")
session = fo.launch_app(view)

### Zero-shot classification

Set `classes` and call `apply_model()`. Each class is embedded as the text `"a photo of a {class}"`, and each image gets the closest one.

In [ ]:
model.classes = ["airplane", "bird", "cat", "cow", "dog", "person", "car"]

dataset.apply_model(model, label_field="eg2_zeroshot", batch_size=16)

session.view = dataset.view()
dataset.count_values("eg2_zeroshot.label")

### Embedding map

`compute_visualization()` projects the stored vectors to 2D with UMAP. Open the Embeddings panel in the App, pick `eg2_viz`, and color the points by `eg2_zeroshot.label`.

In [ ]:
fob.compute_visualization(dataset, embeddings="eg2", brain_key="eg2_viz")

session.view = dataset.view()

### Smaller vectors

The model is trained with Matryoshka dimensions 768, 512, 256 and 128. `embedding_dim` is a property like every other setting (`fps`, `max_frames`, `classes`, ...), so changing it doesn't reload the weights.

In [ ]:
model.embedding_dim = 256
dataset.compute_embeddings(model, embeddings_field="eg2_256", batch_size=16)
model.embedding_dim = 768

## Video with sound: VGGSound

VGGSound is 10-second YouTube clips labeled with one of 309 sound classes. We load 100 clips and a model with both the vision and the audio encoder, so one model handles the frames, the soundtrack, and both together.

In [ ]:
vggsound = fouh.load_from_hub(
    "harpreetsahota/VGGSound", max_samples=100, name="vggsound-eg2"
)

av_model = foz.load_zoo_model(
    MODEL, media_type="video", modalities=["vision", "audio"]
)

### Three embeddings per clip

- `compute_embeddings()` embeds the **frames**: one per second, at most 32, spread across the whole clip.
- `embed_dataset(modality="audio")` embeds the **soundtrack**, resampled to 16 kHz mono.
- `embed_dataset(modality="video+audio")` puts frames and sound into **one input** and returns one vector.

FiftyOne's `compute_embeddings()` accepts only image and video data, so the model brings `embed_dataset()` for audio and for combined inputs. It picks the decoder from the file extension.

In [ ]:
vggsound.compute_embeddings(av_model, embeddings_field="eg2_video")
av_model.embed_dataset(vggsound, "eg2_audio", modality="audio")
av_model.embed_dataset(vggsound, "eg2_av", modality="video+audio")

### Search clips with text

Index the video+audio vectors. Include `modalities` in `model_kwargs` so the App reloads the same configuration.

In [ ]:
fob.compute_similarity(
    vggsound,
    embeddings="eg2_av",
    model=MODEL,
    model_kwargs={"media_type": "video", "modalities": ["vision", "audio"]},
    brain_key="eg2_av_sim",
)

session.view = vggsound.sort_by_similarity("a dog barking", k=10, brain_key="eg2_av_sim")

### Zero-shot: picture, sound, or both

`classify_dataset()` scores stored vectors against the class names and writes a `Classification` per sample, the audio counterpart of `apply_model()`. `evaluate_classifications()` then compares each field to the ground truth.

VGGSound labels mostly name the visible source of a sound ("playing badminton", "playing bassoon"), so the frames carry most of the signal.

In [ ]:
av_model.classes = vggsound.distinct("ground_truth.label")

accuracy = {}
for field in ["eg2_video", "eg2_audio", "eg2_av"]:
    av_model.classify_dataset(vggsound, f"{field}_zs", embeddings_field=field)
    results = vggsound.evaluate_classifications(
        f"{field}_zs", gt_field="ground_truth", eval_key=f"{field}_eval"
    )
    accuracy[field] = results.metrics()["accuracy"]

accuracy

### Embedding map

A UMAP of the video+audio vectors. In the App's Embeddings panel, color the points by `ground_truth.label` or by `eg2_av_eval` to see where zero-shot is right and wrong.

In [ ]:
fob.compute_visualization(vggsound, embeddings="eg2_av", brain_key="eg2_av_viz")

session.view = vggsound.view()

## Video and music: VidMuse-V2M

VidMuse-V2M is a grouped dataset: each group holds a silent music video (`video` slice) and its soundtrack as a wav file (`music` slice). We load 10 groups and embed each slice on its own.

In [ ]:
vidmuse = fouh.load_from_hub(
    "harpreetsahota/VidMuse-V2M", max_samples=20, name="vidmuse-eg2"
)

videos = vidmuse.select_group_slices("video")
music = vidmuse.select_group_slices("music")

videos.compute_embeddings(av_model, embeddings_field="eg2")
av_model.embed_dataset(music, "eg2", modality="audio")

### Find a video's soundtrack

Video and music vectors live in the same space, so a video's embedding can query an index of music. The paired track shares the video's `youtube_id`.

In [ ]:
fob.compute_similarity(
    music,
    embeddings="eg2",
    model=MODEL,
    model_kwargs={"media_type": "audio"},
    brain_key="eg2_music_sim",
)

video = videos.first()
matches = music.sort_by_similarity(video.eg2, k=3, brain_key="eg2_music_sim")

video.youtube_id, matches.values("youtube_id")

### Search music with text

The same index answers text queries.

In [ ]:
music.sort_by_similarity(
    "slow piano music", k=3, brain_key="eg2_music_sim"
).values("youtube_id")

### Embedding maps

One UMAP per slice: FiftyOne can't put the `video` and `music` slices in one view because their media types differ.

In [ ]:
fob.compute_visualization(videos, embeddings="eg2", brain_key="eg2_video_viz")
fob.compute_visualization(music, embeddings="eg2", brain_key="eg2_music_viz")

session.view = videos

## MCAP recordings: VidMuse-V2M-MCAP

The same clips as MCAP files, one per clip, with a `/video` topic (`foxglove.CompressedVideo`, h264) and a `/music` topic (`foxglove.RawAudio`). FiftyOne loads them as `multimodal` samples, which `compute_embeddings()` doesn't accept, so `embed_dataset()` reads the `.mcap` files directly. It finds the video and audio topics by their schema; set `mcap_video_topic` / `mcap_audio_topic` to pick others.

In [ ]:
recordings = fouh.load_from_hub(
    "harpreetsahota/VidMuse-V2M-MCAP", max_samples=10, name="vidmuse-mcap-eg2"
)

av_model.embed_dataset(recordings, "eg2_video", modality="video")
av_model.embed_dataset(recordings, "eg2_audio", modality="audio")
av_model.embed_dataset(recordings, "eg2_av", modality="video+audio")

### MCAP and wav give the same vector

MCAP audio is decoded and resampled exactly like a wav file. Querying the music index from the previous section with a recording's audio vector returns the wav of the same clip.

In [ ]:
recording = recordings.first()
match = music.sort_by_similarity(recording.eg2_audio, k=1, brain_key="eg2_music_sim")

recording.youtube_id, match.values("youtube_id")

### Search recordings with text

Index the video+audio vectors of the recordings like any other dataset.

In [ ]:
fob.compute_similarity(
    recordings,
    embeddings="eg2_av",
    model=MODEL,
    model_kwargs={"media_type": "video", "modalities": ["vision", "audio"]},
    brain_key="eg2_av_sim",
)

session.view = recordings.sort_by_similarity(
    "a music video with a singer", k=3, brain_key="eg2_av_sim"
)

### Embedding map

A UMAP of the recordings' video+audio vectors.

In [ ]:
fob.compute_visualization(recordings, embeddings="eg2_av", brain_key="eg2_av_viz")

session.view = recordings.view()